<a href="https://colab.research.google.com/github/ext-colorful/LangChain-Essentials/blob/main/%F0%9F%A7%B1Building_Blocks%F0%9F%A7%B1L6_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[课程地址](https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388329-lesson-6-memory)😁
[源码地址](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/L1_fast_agent.ipynb)😁
[LANGSMITH官网](https://smith.langchain.com)😁
[LANGCHAIN智能助手](https://chat.langchain.com)😁
[免费的智能体代理商](https://api.chatanywhere.tech)

In Lessons 2–7, you will learn how to use some of the fundamental building blocks in LangChain. These lessons explain and complement create_agent, and you’ll find them useful when creating your own agents. Each lesson is concise and focused.
> 在课程2-7中，你将学习如何使用LangChain中的一些基本构建模块。这些课程解释并补充了create_agent，当你创建自己的代理时，你会发现它们很有用。每个课程都简洁而专注。

### Learn how to give your agent the ability to maintain state between invocations.
> 学习如何让你的代理在调用之间保持状态。

# Memory

![链接文字](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/assets/LC_Memory_before.png?raw=true)

Persisting messages, or 'agent state' between invocations of the agent.
> 在代理的调用之间持久化消息，或称为'代理状态'

## Setup
Load and/or check for needed environmental variables
> 加载和/或检查所需的环境变量

In [ ]:
!pip install -q langgraph==1.0.3 langchain==1.0.8 langchain-openai==1.0.3 langchain-community==0.4.1 langgraph-cli[inmem]==0.4.7 langchain-mcp-adapters==0.1.13
# Used to securely store your API key
# 用于安全存储您的API密钥
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('OPENAI_API_BASE')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain_agent_L6_memory"

In [ ]:
!git clone https://github.com/langchain-ai/lca-langchainV1-essentials.git
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///lca-langchainV1-essentials/python/Chinook.db")

fatal: destination path 'lca-langchainV1-essentials' already exists and is not an empty directory.


In [ ]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [ ]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [ ]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_GfhCOzj1A2GYx9EH6DTpO1xe)
 Call ID: call_GfhCOzj1A2GYx9EH6DTpO1xe
  Args:
    query: SELECT invoice_id, total FROM invoices WHERE customer_name = 'Frank Harris' ORDER BY invoice_date DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: invoices
[SQL: SELECT invoice_id, total FROM invoices WHERE customer_name = 'Frank Harris' ORDER BY invoice_date DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_0zen5pSBSDGJEF32LQsKBKU7)
 Call ID: call_0zen5pSBSDGJEF32LQsKBKU7
  Args:
    query: SELECT name

In [ ]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_Sk2p0NnyhToEZh5Ocg9Hs2Oy)
 Call ID: call_Sk2p0NnyhToEZh5Ocg9Hs2Oy
  Args:
    query: SELECT title FROM sqlite_master WHERE type='table' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such column: title
[SQL: SELECT title FROM sqlite_master WHERE type='table' LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_tKwVTi7dnxsDGKH4zgPo8eNj)
 Call ID: call_tKwVTi7dnxsDGKH4zgPo8eNj
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' LIMIT 5;
================================= Tool Message =================================
Name: execute_sq

## Add memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [ ]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_x4JdZvhaHFACfeB8JupBq4Mi)
 Call ID: call_x4JdZvhaHFACfeB8JupBq4Mi
  Args:
    query: SELECT id FROM customers WHERE name = 'Frank Harris' LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: customers
[SQL: SELECT id FROM customers WHERE name = 'Frank Harris' LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_xKccbh8wZroyD4qk4tb0Mw8q)
 Call ID: call_xKccbh8wZroyD4qk4tb0Mw8q
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message ============

In [ ]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_VSh7TYA1eU4RMSBMuDuii3A8)
 Call ID: call_VSh7TYA1eU4RMSBMuDuii3A8
  Args:
    query: SELECT il.InvoiceLineId, t.Name FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374;
================================= Tool Message =================================
Name: execute_sql

[(2021, 'Holier Than Thou'), (2022, 'Through The Never'), (2023, 'My Friend Of Misery'), (2024, 'The Wait'), (2025, 'Blitzkrieg'), (2026, 'So What')]
================================== Ai Message ==================================

The titles on your last invoice were: Holier Than Thou, Through The Never, My Friend Of Misery, The Wait, Blitzkrieg, and So What.


## Try your own queries 尝试你自己的查询

Now that there is memory, check the agents recall!
> 既然有了记忆，检查一下代理的回忆能力吧！

In [ ]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?
================================== Ai Message ==================================

How can I assist you further with your invoices or any other information?
